In [21]:
import pandas as pd
import json

In [22]:
colunas_localizacao = [
    # Localização
    'SG_UF_PROVA',
    'SG_REGIAO',
]

colunas_geral = [
    'SG_UF_PROVA',
    'SG_REGIAO',

    # Presença
    'TP_PRESENCA_CN', 
    'TP_PRESENCA_CH', 
    'TP_PRESENCA_LC', 
    'TP_PRESENCA_MT',
    'TP_PRESENCA_GERAL',
    'TP_PRESENCA_REDACAO',

    # Notas
    'NU_NOTA_CN', 
    'NU_NOTA_CH', 
    'NU_NOTA_LC', 
    'NU_NOTA_MT',
    'NU_NOTA_REDACAO'
    ]

colunas_aspectos_sociais = [
    'SG_UF_PROVA',
    'SG_REGIAO',

    # Variáveis sociais
    'TP_SEXO',
    'TP_COR_RACA',
    'TP_ESTADO_CIVIL',
    'TP_FAIXA_ETARIA',
    'TP_ST_CONCLUSAO',
    'TP_DEPENDENCIA_ADM_ESC',
    'TP_ESCOLA',
    'TP_ENSINO',
    'TP_LOCALIZACAO_ESC',
    'TP_NACIONALIDADE',
    
    
    # Questões socioeconômicas
    'Q001', 
    'Q002', 
    'Q005', 
    'TP_FAIXA_SALARIAL', 
    'Q025',
    
    # Infraestrutura
    'NU_INFRAESTRUTURA'
    ]

colunas_desempenho = [
    'SG_UF_PROVA',
    'SG_REGIAO',
    
    # Características do candidato
    'TP_SEXO',
    'TP_COR_RACA',
    'TP_DEPENDENCIA_ADM_ESC',
    'TP_ST_CONCLUSAO',
    'TP_FAIXA_ETARIA',
    'TP_ESTADO_CIVIL',
    'TP_ESCOLA',
    'TP_ENSINO',
    'TP_LOCALIZACAO_ESC',
    'TP_NACIONALIDADE',
    'NU_INFRAESTRUTURA',
    
    # Notas
    'NU_NOTA_CN', 
    'NU_NOTA_CH', 
    'NU_NOTA_LC', 
    'NU_NOTA_MT',
    'NU_NOTA_REDACAO',
    
    # Categorias de desempenho
    'NU_DESEMPENHO',

    # Questões socioeconômicas
    'Q001',
    'Q002',
    'Q005',
    'TP_FAIXA_SALARIAL',
    'Q025'
    ]

In [23]:
# Carregar microdados completos
arquivo = '../../arquivos-iniciacao/microdados_tratado.parquet'
arquivo_dtypes = '../../arquivos-iniciacao/dtypes.json'

# Especificar explicitamente o engine
microdados = pd.read_parquet(arquivo, engine='pyarrow')

dtypes = pd.read_json(arquivo_dtypes, orient='index', typ='series')

microdados = microdados.astype(dtypes)


In [24]:
# microdados = microdados[microdados['SG_REGIAO'].isin(['Sul', 'Sudeste', 'Centro-Oeste'])]

In [25]:
# Colunas desempenho sem notas zeros
microdados_desempenho = microdados.copy()

microdados_desempenho = microdados_desempenho[(microdados_desempenho['TP_PRESENCA_GERAL'] == 3) & (microdados_desempenho['NU_MEDIA_GERAL'] != -1)]

microdados_desempenho = microdados_desempenho[microdados_desempenho['NU_MEDIA_GERAL'] != -1]

In [26]:
import numpy as np

def otimizar_para_parquet(df, tem_notas=False):
    """
    Otimiza dtypes ANTES de salvar no parquet para reduzir tamanho em disco
    e eliminar conversões em tempo real no data_loader.
    
    - Notas: replace -1 → NaN, divide por 10, cast float32
    - int64 com range pequeno → int8
    - float64 → float32
    - Preserva category e int16
    """
    df = df.copy()
    
    # Notas: aplicar transformação semântica (antes era feita no data_loader)
    if tem_notas:
        notas_cols = [c for c in df.columns if c.startswith('NU_NOTA_')]
        for col in notas_cols:
            df[col] = df[col].replace(-1, np.nan).astype('float64') / 10.0
            df[col] = df[col].astype('float32')
    
    # Downcast int64 → int8 para colunas com range pequeno
    for col in df.select_dtypes(include=['int64', 'int32']).columns:
        vmin, vmax = df[col].min(), df[col].max()
        if vmin >= -128 and vmax <= 127:
            df[col] = df[col].astype('int8')
        elif vmin >= -32768 and vmax <= 32767:
            df[col] = df[col].astype('int16')
    
    # Downcast float64 → float32
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    
    return df

# Aplicar otimização a cada dataset
df_localizacao = microdados[colunas_localizacao]
df_geral = otimizar_para_parquet(microdados[colunas_geral], tem_notas=True)
df_aspectos = otimizar_para_parquet(microdados[colunas_aspectos_sociais], tem_notas=False)
df_desempenho = otimizar_para_parquet(microdados_desempenho[colunas_desempenho], tem_notas=True)

print("Memória após otimização:")
for nome, df in [("localizacao", df_localizacao), ("geral", df_geral), 
                  ("aspectos_sociais", df_aspectos), ("desempenho", df_desempenho)]:
    mem = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"  {nome:25s} {len(df):>10,} rows  {mem:6.1f} MB")

Memória após otimização:
  localizacao                3,933,955 rows     7.5 MB
  geral                      3,933,955 rows   105.1 MB
  aspectos_sociais           3,933,955 rows    67.5 MB
  desempenho                 2,678,264 rows   120.1 MB


In [27]:
# Salvar parquets já otimizados (dtypes finais, notas já divididas por 10)
df_localizacao.to_parquet('sample_localizacao.parquet', index=False, engine='pyarrow')
df_geral.to_parquet('sample_geral.parquet', index=False, engine='pyarrow')
df_aspectos.to_parquet('sample_aspectos_sociais.parquet', index=False, engine='pyarrow')
df_desempenho.to_parquet('sample_desempenho.parquet', index=False, engine='pyarrow')

# Salvar dtypes dos DataFrames JÁ OTIMIZADOS (refletem os tipos finais)
for nome, df_opt in [("localizacao", df_localizacao), ("geral", df_geral), 
                      ("aspectos_sociais", df_aspectos), ("desempenho", df_desempenho)]:
    dtypes_dict = {col: str(df_opt[col].dtype) for col in df_opt.columns}
    with open(f'dtypes_{nome}.json', 'w') as f:
        json.dump(dtypes_dict, f)

# Manter dtypes.json geral para referência
with open('dtypes.json', 'w') as f:
    json.dump(dtypes.to_dict(), f)

print("Arquivos salvos:")
import os
for f in sorted(os.listdir('.')):
    if f.endswith(('.parquet', '.json')):
        size = os.path.getsize(f) / 1024 / 1024
        print(f"  {f:45s} {size:6.1f} MB")

Arquivos salvos:
  dtypes.json                                      0.0 MB
  dtypes_aspectos_sociais.json                     0.0 MB
  dtypes_desempenho.json                           0.0 MB
  dtypes_geral.json                                0.0 MB
  dtypes_localizacao.json                          0.0 MB
  sample_aspectos_sociais.parquet                 22.3 MB
  sample_desempenho.parquet                       34.6 MB
  sample_geral.parquet                            36.1 MB
  sample_localizacao.parquet                       3.8 MB


In [28]:
microdados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3933955 entries, 0 to 3933954
Data columns (total 31 columns):
 #   Column                  Dtype   
---  ------                  -----   
 0   TP_FAIXA_ETARIA         category
 1   TP_SEXO                 category
 2   TP_ESTADO_CIVIL         category
 3   TP_COR_RACA             category
 4   TP_NACIONALIDADE        category
 5   TP_ST_CONCLUSAO         category
 6   TP_ESCOLA               category
 7   TP_ENSINO               category
 8   TP_DEPENDENCIA_ADM_ESC  category
 9   TP_LOCALIZACAO_ESC      category
 10  SG_UF_PROVA             category
 11  TP_PRESENCA_CN          category
 12  TP_PRESENCA_CH          category
 13  TP_PRESENCA_LC          category
 14  TP_PRESENCA_MT          category
 15  NU_NOTA_CN              int16   
 16  NU_NOTA_CH              int16   
 17  NU_NOTA_LC              int16   
 18  NU_NOTA_MT              int16   
 19  NU_NOTA_REDACAO         int16   
 20  Q001                    category
 21  Q002    